# Clase 9 — Regresión y Baseline del Proyecto

Esta es la última clase del Módulo 2. Tiene dos partes: primero, aprendemos a resolver problemas de **regresión** (predecir un número en lugar de una categoría). Después, aplicamos todo lo del módulo para documentar el **baseline del proyecto integrador**, que une ML con prompting.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Regresión vs. clasificación |
| 2 | El dataset: precios de autos |
| 3 | Modelo lineal y métricas de regresión |
| 4 | Ridge y Lasso: regularización |
| 5 | Detectar sobreajuste |
| 6 | Proyecto integrador: definir el baseline |
| 7 | Entregable del Módulo 2 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Librerías listas.")

---
## 1. Regresión vs. clasificación

| | Clasificación | Regresión |
|---|---|---|
| **Salida del modelo** | Categoría discreta | Número continuo |
| **Pregunta típica** | ¿Es spam o no? | ¿Cuánto vale? |
| **Métrica principal** | Accuracy / F1 | MSE / RMSE / R² |
| **Ejemplo** | ¿El cliente va a churnar? | ¿Cuánto va a consumir? |

> 💡 El flujo de trabajo es idéntico: cargar datos → split → preprocesar → entrenar → evaluar. Solo cambian el tipo de modelo y las métricas de evaluación.

---
## 2. El dataset: precios de casas en California

El dataset California Housing contiene datos de 20,640 zonas del estado de California. La tarea es predecir el valor mediano de las casas (`MedHouseVal`, en cientos de miles de dólares).

In [ ]:
# ─── Cargar el dataset ────────────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Shape: {df.shape}")
print()
print("Descripción de las variables:")

descripciones = {
    "MedInc":      "Ingreso mediano del hogar (en 10k USD)",
    "HouseAge":    "Edad mediana de las casas",
    "AveRooms":    "Promedio de habitaciones por hogar",
    "AveBedrms":   "Promedio de dormitorios por hogar",
    "Population":  "Población del bloque",
    "AveOccup":    "Promedio de ocupantes por hogar",
    "Latitude":    "Latitud del bloque",
    "Longitude":   "Longitud del bloque",
    "MedHouseVal": "Valor mediano de la casa — ETIQUETA (objetivo)",
}
for col, desc in descripciones.items():
    print(f"  {col:<15}: {desc}")

In [ ]:
# ─── Exploración básica ───────────────────────────────────────────────────────
print("Estadísticas descriptivas:")
df.describe().round(2)

In [ ]:
# ─── Distribución del target ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df["MedHouseVal"], bins=50, color="steelblue", edgecolor="white")
ax1.set(xlabel="Valor mediano (100k USD)", ylabel="Frecuencia",
        title="Distribución del target")

ax2.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.1, s=3, color="coral")
ax2.set(xlabel="Ingreso mediano", ylabel="Valor casa",
        title="Ingreso vs. Precio (feature más correlacionada)")

plt.tight_layout()
plt.show()

In [ ]:
# ─── Split ────────────────────────────────────────────────────────────────────
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} — Test: {X_test.shape[0]}")

---
## 3. Modelo lineal y métricas de regresión

In [ ]:
# ─── Pipeline: scaler + regresión lineal ─────────────────────────────────────
pipeline_lr = Pipeline([
    ("escalar", StandardScaler()),
    ("modelo",  LinearRegression()),
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

print("Métricas — Regresión Lineal:")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_lr):.4f}  (error absoluto promedio en 100k USD)")
print(f"  RMSE: {mean_squared_error(y_test, y_pred_lr, squared=False):.4f}  (igual que MAE pero penaliza errores grandes)")
print(f"  R²:   {r2_score(y_test, y_pred_lr):.4f}  (1.0 = perfecto, 0 = tan malo como la media)")

| Métrica | Fórmula | Qué mide |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Error promedio en las mismas unidades que el target |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Error promedio con mayor peso a los errores grandes |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporción de la varianza explicada por el modelo |

In [ ]:
# ─── Visualizar predicciones vs. valores reales ───────────────────────────────
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_lr, alpha=0.3, s=5, color="steelblue")

# Línea perfecta: si el modelo fuera perfecto, todos los puntos caerían acá
lim = [y_test.min(), y_test.max()]
plt.plot(lim, lim, color="red", linewidth=1, label="Predicción perfecta")

plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión Lineal: real vs. predicho")
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Ridge y Lasso: regularización

La regresión lineal simple puede sobreajustar cuando hay muchas features correlacionadas. **Ridge** y **Lasso** agregan una penalización a los coeficientes para controlar este efecto.

| Modelo | Penalización | Efecto |
|---|---|---|
| **Ridge** | $\lambda \sum w_i^2$ | Reduce todos los coeficientes hacia cero, sin eliminar ninguno |
| **Lasso** | $\lambda \sum |w_i|$ | Puede llevar coeficientes exactamente a cero → selección de features |

In [ ]:
# ─── Comparar Ridge, Lasso y Linear ──────────────────────────────────────────
modelos_reg = {
    "Linear":        LinearRegression(),
    "Ridge (α=1)": Ridge(alpha=1.0),
    "Lasso (α=0.1)": Lasso(alpha=0.1, max_iter=5000),
}

resultados_reg = []
for nombre, modelo in modelos_reg.items():
    pipe = Pipeline([("escalar", StandardScaler()), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados_reg.append({
        "modelo":  nombre,
        "MAE":     mean_absolute_error(y_test, y_pred),
        "RMSE":    mean_squared_error(y_test, y_pred, squared=False),
        "R²":      r2_score(y_test, y_pred),
    })

df_reg = pd.DataFrame(resultados_reg).set_index("modelo")
df_reg.round(4)

---
## 5. Detectar sobreajuste

El **sobreajuste (overfitting)** ocurre cuando el modelo aprende los datos de train demasiado bien y no generaliza. Se detecta comparando el score en train vs. test: si el train es mucho mejor, hay overfitting.

In [ ]:
# ─── Train vs. Test score para detectar overfitting ──────────────────────────
print("R² en train vs. test:")
print()

for nombre, modelo in modelos_reg.items():
    pipe = Pipeline([("escalar", StandardScaler()), ("modelo", modelo)])
    pipe.fit(X_train, y_train)

    r2_train = r2_score(y_train, pipe.predict(X_train))
    r2_test  = r2_score(y_test,  pipe.predict(X_test))
    diferencia = r2_train - r2_test

    flag = "⚠️  posible overfitting" if diferencia > 0.05 else "✓"
    print(f"  {nombre:<18}  train={r2_train:.4f}  test={r2_test:.4f}  diff={diferencia:.4f}  {flag}")

> 💡 En modelos lineales con datasets grandes, el overfitting es raro. Aparece más frecuentemente en Decision Trees sin profundidad limitada o en modelos con muchas features y pocos datos.

---
## 6. Proyecto integrador: definir el baseline

El Módulo 2 tiene dos líneas que ahora unimos:
- **Prompting** (clases 1–5): diseñar, documentar y versionar prompts efectivos
- **ML básico** (clases 6–9): entrenar y evaluar modelos de ML con scikit-learn

El proyecto integrador es un **sistema de predicción simple + guía de prompts**:
1. Un modelo de ML que resuelve un problema de clasificación o regresión
2. Una guía de prompts que asiste al análisis (explicar predicciones, resumir resultados, generar reportes)

> 💡 En los módulos siguientes vamos a escalar esto: fine-tuning, RAG, y sistemas con LLMs más poderosos. El baseline de hoy es la base sobre la que vamos a construir.

In [ ]:
# ─── Plantilla de documentación del baseline ─────────────────────────────────
# Completá cada campo con los datos de tu proyecto.

baseline = {
    "nombre_proyecto": "[TU PROYECTO AQUÍ]",

    # ─── Parte ML ────────────────────────────────────────────────────────────
    "tipo_problema":   "clasificacion | regresion",   # elegí uno
    "dataset":         "[nombre o descripción del dataset]",
    "n_ejemplos":      0,
    "n_features":      0,
    "variable_target": "[nombre de la variable a predecir]",

    "modelo_baseline": "[modelo elegido, ej: LogisticRegression]",
    "metrica_principal": "[accuracy / F1 / RMSE / R²]",
    "score_baseline_test": 0.0,
    "score_baseline_train": 0.0,

    # ─── Parte Prompting ────────────────────────────────────────────────────
    "tarea_prompting":        "[qué tarea asiste el LLM, ej: explicar predicciones]",
    "modelo_llm":             "gemini-2.5-flash-lite",
    "prompt_baseline_id":     "[ID del prompt en el repositorio de la clase 5]",
    "prompt_baseline_score":  "[resultado cualitativo: malo / aceptable / bueno]",

    # ─── Meta ────────────────────────────────────────────────────────────────
    "alumno":    "[tu nombre]",
    "fecha":     pd.Timestamp.now().strftime("%Y-%m-%d"),
}

# Mostrar como tabla
df_baseline = pd.DataFrame(
    [{"campo": k, "valor": str(v)} for k, v in baseline.items()]
).set_index("campo")

print("BASELINE DEL PROYECTO — Módulo 2")
print("="*50)
print(df_baseline.to_string())

In [ ]:
# ─── Guardar el baseline en JSON ──────────────────────────────────────────────
import json

ruta_baseline = "baseline_proyecto_m2.json"
with open(ruta_baseline, "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)

print(f"Baseline guardado en: {ruta_baseline}")
print("Este archivo es el punto de partida para los módulos siguientes.")

---
## 7. Entregable del Módulo 2

In [ ]:
# TODO: Completá el proyecto integrador del módulo.
#
# Parte A — ML:
#   1. Elegí un dataset propio o de sklearn/seaborn
#   2. Aplicá el flujo completo: split → pipeline → entrená 2+ modelos → elegí el mejor
#   3. Documentá las métricas en la plantilla baseline de la sección 6
#
# Parte B — Prompting:
#   4. Definí una tarea donde el LLM ayude con el análisis
#      (por ejemplo: "explicar en lenguaje natural por qué el modelo predijo X")
#   5. Diseñá al menos 2 versiones del prompt usando las técnicas del módulo
#      (few-shot, CoT, o template con variables)
#   6. Agregá ambos prompts al repositorio de la clase 5 con sus metadatos
#
# Entregable final: este notebook ejecutado + baseline_proyecto_m2.json + prompt_repositorio.json

print("¡Completá el proyecto integrador y entregá los tres archivos!")

---
## Cierre del Módulo 2

Completaste el Módulo 2 del programa. Esto es lo que construiste:

| Clase | Lo que aprendiste | Lo que podés hacer |
|---|---|---|
| 1 | Anatomía del prompt | Diseñar prompts estructurados con 5 componentes |
| 2 | Anti-patrones | Diagnosticar y reformular prompts rotos |
| 3 | Few-shot / Zero-shot | Guiar al LLM con ejemplos para tareas específicas |
| 4 | Chain-of-Thought | Descomponer razonamientos complejos en pasos |
| 5 | Repositorio de prompts | Documentar, versionar y reutilizar prompts |
| 6 | Intro ML | Entrenar y evaluar un modelo de clasificación básico |
| 7 | Pipeline ML | Construir un flujo reproducible con preprocesamiento |
| 8 | Clasificación | Comparar algoritmos y elegir con criterio |
| 9 | Regresión + Baseline | Predecir valores continuos y documentar el punto de partida |

**Módulo 3:** Embeddings, búsqueda semántica y sistemas RAG (Retrieval-Augmented Generation).